# CDC Open Data Accessor - Example Notebook

This notebook demonstrates how to use the CDC Open Data accessor to fetch public health surveillance data from the Centers for Disease Control and Prevention (CDC).

## Overview

The CDC Open Data Portal (data.cdc.gov) provides access to:
- COVID-19 surveillance data
- Influenza surveillance
- HIV/AIDS surveillance
- Chronic Disease Indicators
- NNDSS (National Notifiable Diseases Surveillance System)
- Vaccination coverage data

**Data Source:** https://data.cdc.gov/
**API:** Socrata Open Data API (SODA)

In [ ]:
# Import necessary libraries

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta

from epidatasets.sources.cdc_opendata import CDCOpenDataAccessor, get_cdc_datasets, get_cdc_covid

# Set plotting style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("✅ Libraries imported successfully!")

## 1. Initialize CDC Open Data Accessor

In [ ]:
# Initialize the accessor
cdc = CDCOpenDataAccessor()

print("✅ CDC Open Data accessor initialized")
print(f"Cache directory: {cdc.cache_dir}")

## 2. Explore Available Datasets

In [ ]:
# Get all available datasets
datasets = cdc.get_available_datasets()

print(f"Total datasets available: {len(datasets)}")
print("\nDatasets by category:")
print(datasets['category'].value_counts())

# Show all datasets
datasets

In [ ]:
# Filter by category
infectious_disease = cdc.get_available_datasets(category="infectious_disease")
print("Infectious disease datasets:")
infectious_disease[['dataset_key', 'name', 'update_frequency']]

## 3. COVID-19 Data

In [ ]:
# Get COVID-19 data for California
covid_ca = cdc.get_covid_cases(
    state="CA",
    start_date="2024-01-01",
    limit=1000
)

print(f"Retrieved {len(covid_ca)} records")
print("\nFirst 5 records:")
covid_ca.head()

In [ ]:
# Plot COVID-19 trends for California
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8))

# Daily new cases
ax1.plot(covid_ca['submission_date'], covid_ca['new_case'], marker='o', linewidth=2, markersize=3)
ax1.set_title('COVID-19 Daily New Cases - California (2024)', fontsize=14, fontweight='bold')
ax1.set_ylabel('New Cases')
ax1.grid(True, alpha=0.3)

# Daily new deaths
ax2.plot(covid_ca['submission_date'], covid_ca['new_death'], 
         marker='s', linewidth=2, color='red', markersize=3)
ax2.set_title('COVID-19 Daily New Deaths - California (2024)', fontsize=14, fontweight='bold')
ax2.set_ylabel('New Deaths')
ax2.set_xlabel('Date')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Compare multiple states
states = ["CA", "NY", "TX", "FL"]
covid_comparison = []

for state in states:
    try:
        data = cdc.get_covid_cases(
            state=state,
            start_date="2024-01-01",
            limit=500
        )
        data['state_name'] = state
        covid_comparison.append(data)
    except Exception as e:
        print(f"Error fetching {state}: {e}")

if covid_comparison:
    covid_all = pd.concat(covid_comparison, ignore_index=True)
    print(f"\nTotal records: {len(covid_all)}")

In [ ]:
# Plot comparison
if covid_comparison:
    plt.figure(figsize=(12, 6))
    
    for state in states:
        state_data = covid_all[covid_all['state_name'] == state]
        plt.plot(state_data['submission_date'], state_data['new_case'], 
                marker='o', label=state, linewidth=2, markersize=3)
    
    plt.title('COVID-19 Daily New Cases - Multi-State Comparison (2024)', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Date')
    plt.ylabel('New Cases')
    plt.legend(title='State')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 4. Influenza Surveillance Data

In [ ]:
# Get influenza data for New York
flu_ny = cdc.get_influenza_data(
    state="NY",
    season="2023-24",
    limit=1000
)

print(f"Retrieved {len(flu_ny)} records")
print("\nColumn names:")
print(flu_ny.columns.tolist())

flu_ny.head()

In [ ]:
# Plot influenza activity
if 'total_specimens' in flu_ny.columns:
    plt.figure(figsize=(12, 6))
    plt.plot(flu_ny['weekendingdate'], flu_ny['total_specimens'], 
             marker='o', linewidth=2, label='Total Specimens')
    plt.plot(flu_ny['weekendingdate'], flu_ny['total_a'], 
             marker='s', linewidth=2, label='Influenza A')
    plt.plot(flu_ny['weekendingdate'], flu_ny['total_b'], 
             marker='^', linewidth=2, label='Influenza B')
    
    plt.title('Influenza Surveillance - New York (2023-24 Season)', 
              fontsize=14, fontweight='bold')
    plt.xlabel('Week Ending Date')
    plt.ylabel('Number of Specimens')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 5. Chronic Disease Indicators

In [ ]:
# Get diabetes prevalence data
diabetes = cdc.get_chronic_disease_indicator(
    indicator="DIABETES",
    year=2021,
    limit=1000
)

print(f"Retrieved {len(diabetes)} records")

# Show unique stratifications
if 'stratification1' in diabetes.columns:
    print("\nAvailable stratifications:")
    print(diabetes['stratification1'].unique())

diabetes.head()

In [ ]:
# Compare diabetes prevalence by state
diabetes_states = cdc.get_chronic_disease_indicator(
    indicator="DIABETES",
    year=2021,
    stratification="Overall",
    limit=1000
)

# Filter for state-level data
if 'locationabbr' in diabetes_states.columns:
    diabetes_states = diabetes_states[diabetes_states['locationabbr'] != 'US']
    
    # Sort by data value
    if 'datavalue' in diabetes_states.columns:
        diabetes_states['datavalue'] = pd.to_numeric(diabetes_states['datavalue'], errors='coerce')
        diabetes_top = diabetes_states.nlargest(10, 'datavalue')
        
        # Plot top 10 states
        plt.figure(figsize=(10, 6))
        plt.barh(diabetes_top['locationabbr'], diabetes_top['datavalue'])
        plt.title('Top 10 States - Diabetes Prevalence (2021)', 
                  fontsize=14, fontweight='bold')
        plt.xlabel('Prevalence (%)')
        plt.ylabel('State')
        plt.tight_layout()
        plt.show()

## 6. NNDSS - National Notifiable Diseases Surveillance System

In [ ]:
# List available notifiable diseases
diseases = cdc.list_notifiable_diseases()
print(f"Total notifiable diseases: {len(diseases)}")
print("\nFirst 20 diseases:")
for i, disease in enumerate(diseases[:20], 1):
    print(f"{i:2d}. {disease}")

In [ ]:
# Get NNDSS data for a specific disease
lyme_data = cdc.get_nndss_data(
    disease="Lyme disease",
    year=2023,
    limit=500
)

print(f"Retrieved {len(lyme_data)} records")

if not lyme_data.empty:
    print("\nFirst 5 records:")
    print(lyme_data.head())
    
    # Show case counts by state
    if 'state' in lyme_data.columns and 'cases' in lyme_data.columns:
        lyme_by_state = lyme_data.groupby('state')['cases'].sum().sort_values(ascending=False)
        print("\nTop 10 states by Lyme disease cases (2023):")
        print(lyme_by_state.head(10))

## 7. HIV Surveillance Data

In [ ]:
# Get HIV data for a specific state
hiv_ny = cdc.get_hiv_data(
    state="NY",
    year=2021,
    limit=500
)

print(f"Retrieved {len(hiv_ny)} records")

if not hiv_ny.empty:
    print("\nColumn names:")
    print(hiv_ny.columns.tolist())
    print("\nFirst 5 records:")
    hiv_ny.head()

## 8. Summary and Export

In [ ]:
# Get dataset info
dataset_info = cdc.get_dataset_info("covid_cases_deaths")

print("Dataset Information:")
print("="*60)
for key, value in dataset_info.items():
    print(f"{key.replace('_', ' ').title()}: {value}")

In [ ]:
# Export data to CSV
if not covid_ca.empty:
    output_file = "cdc_covid_california_2024.csv"
    covid_ca.to_csv(output_file, index=False)
    print(f"✅ Data exported to: {output_file}")
    
    from pathlib import Path
    file_size = Path(output_file).stat().st_size / 1024
    print(f"File size: {file_size:.2f} KB")

## Summary

This notebook demonstrated:

1. ✅ Initializing the CDC Open Data accessor
2. ✅ Exploring available datasets (15+ datasets across 6 categories)
3. ✅ Fetching COVID-19 data by state
4. ✅ Comparing data across multiple states
5. ✅ Accessing influenza surveillance data
6. ✅ Retrieving Chronic Disease Indicators
7. ✅ Querying NNDSS notifiable disease data
8. ✅ Exporting data for further analysis

### Available Data Categories

- **Infectious Diseases**: COVID-19, Influenza, HIV/AIDS, Hepatitis, NNDSS
- **Chronic Diseases**: Diabetes, heart disease, cancer indicators
- **Behavioral**: BRFSS surveillance data
- **Mortality**: Underlying cause of death
- **Environmental**: Environmental health tracking
- **Vaccination**: Coverage data

### Resources

- CDC Open Data Portal: https://data.cdc.gov/
- Socrata API Docs: https://dev.socrata.com/
- CDC Data Catalogue: https://data.cdc.gov/browse